#📘 Lab Assignment
Create a Transformer from Scratch using PyTorch

✅ STEP 1: Import Libraries

In [1]:
# Import PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

✅ STEP 2: Create Positional Encoding

Transformer has no recurrence, so we must encode position information.

In [2]:
class PositionalEncoding(nn.Module):
    """
    Adds positional information to input embeddings
    """
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        # Create matrix of shape (max_len, d_model)
        pe = torch.zeros(max_len, d_model)

        # Position vector (0,1,2,3,...)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # Compute division term
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                             (-math.log(10000.0) / d_model))

        # Apply sin to even indices
        pe[:, 0::2] = torch.sin(position * div_term)

        # Apply cos to odd indices
        pe[:, 1::2] = torch.cos(position * div_term)

        # Add batch dimension
        pe = pe.unsqueeze(0)

        # Register as buffer (not trainable)
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        Add positional encoding to input
        """
        x = x + self.pe[:, :x.size(1)]
        return x

✅ STEP 3: Scaled Dot Product Attention

Core idea of transformer.

In [3]:
def scaled_dot_product_attention(Q, K, V):
    """
    Computes Attention(Q,K,V)
    """

    # Compute dot product between Q and K
    scores = torch.matmul(Q, K.transpose(-2, -1))

    # Scale by sqrt(d_k)
    d_k = Q.size(-1)
    scores = scores / math.sqrt(d_k)

    # Apply softmax
    attention_weights = F.softmax(scores, dim=-1)

    # Multiply by V
    output = torch.matmul(attention_weights, V)

    return output, attention_weights

✅ STEP 4: Multi-Head Attention

In [4]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention Layer
    """
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()

        self.num_heads = num_heads
        self.d_model = d_model

        assert d_model % num_heads == 0

        self.d_k = d_model // num_heads

        # Linear layers for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.fc = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V):

        batch_size = Q.size(0)

        # Linear projections
        Q = self.W_q(Q)
        K = self.W_k(K)
        V = self.W_v(V)

        # Split into multiple heads
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)

        # Apply attention
        output, attention = scaled_dot_product_attention(Q, K, V)

        # Concatenate heads
        output = output.transpose(1,2).contiguous().view(batch_size, -1, self.d_model)

        return self.fc(output)

✅ STEP 5: Feed Forward Network

In [5]:
class FeedForward(nn.Module):
    """
    Position-wise Feed Forward Network
    """
    def __init__(self, d_model, d_ff=2048):
        super(FeedForward, self).__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

✅ STEP 6: Encoder Layer

In [6]:
class EncoderLayer(nn.Module):
    """
    Single Transformer Encoder Layer
    """
    def __init__(self, d_model, num_heads):
        super(EncoderLayer, self).__init__()

        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)

        self.ff = FeedForward(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):

        # Multi-head attention
        attn_output = self.attention(x, x, x)

        # Add & Normalize
        x = self.norm1(x + attn_output)

        # Feed Forward
        ff_output = self.ff(x)

        # Add & Normalize
        x = self.norm2(x + ff_output)

        return x

✅ STEP 7: Complete Transformer Encoder

In [7]:
class Transformer(nn.Module):
    """
    Complete Transformer Encoder
    """
    def __init__(self, vocab_size, d_model, num_heads, num_layers):
        super(Transformer, self).__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model)

        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads) for _ in range(num_layers)]
        )

        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):

        # Embedding
        x = self.embedding(x)

        # Add positional encoding
        x = self.pos_encoding(x)

        # Pass through encoder layers
        for layer in self.layers:
            x = layer(x)

        # Final output layer
        return self.fc_out(x)

### Test the Transformer Model

Let's create a dummy input to test the `Transformer` model and observe its output.

In [8]:
# Define model parameters
vocab_size = 1000
d_model = 512
num_heads = 8
num_layers = 6

# Instantiate the Transformer model
model = Transformer(vocab_size, d_model, num_heads, num_layers)

# Create a dummy input tensor
# batch_size, sequence_length
batch_size = 2
sequence_length = 10
dummy_input = torch.randint(0, vocab_size, (batch_size, sequence_length))

print(f"Dummy input shape: {dummy_input.shape}")

# Pass the dummy input through the model
output = model(dummy_input)

print(f"\nOutput shape: {output.shape}")
print("\nSample output (first item in batch, first token, first 5 dimensions):")
print(output[0, 0, :5])

Dummy input shape: torch.Size([2, 10])

Output shape: torch.Size([2, 10, 1000])

Sample output (first item in batch, first token, first 5 dimensions):
tensor([ 0.3721,  0.1851, -0.1767, -0.1024,  0.3648], grad_fn=<SliceBackward0>)
